In [1]:
# Import necessari
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import dill

# Import di LIME per l'interpretazione dei modelli
from lime.lime_tabular import LimeTabularExplainer

# Rimozione dei warnings per rendere l'output più pulito
import warnings
warnings.filterwarnings('ignore')

SEED = 2025

In [2]:
# Caricamento e preparazione dei dati
data = load_breast_cancer()
x = data.data
y = data.target

# Divisione dei dati in set di addestramento e test
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=SEED)

# Normalizzazione dei dati
scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain)
xtest = scaler.transform(xtest)

In [3]:
# Creazione e addestramento del MLP
mlp = MLPClassifier(hidden_layer_sizes=(50,), activation='relu', solver='adam', max_iter=1000, random_state=42)
mlp.fit(xtrain, ytrain)

# Valutazione dell'accuratezza del modello
train_accuracy = accuracy_score(ytrain, mlp.predict(xtrain))
test_accuracy = accuracy_score(ytest, mlp.predict(xtest))
print(f"Accuratezza train: {train_accuracy:.4f}, Accuratezza test: {test_accuracy:.4f}")

Accuratezza train: 0.9956, Accuratezza test: 0.9737


In [13]:
# Creazione dell'explainer LIME specificando tutti i parametri richiesti
explainer = LimeTabularExplainer(xtrain,
                                 mode='classification',
                                 training_labels=ytrain,
                                 feature_names=data.feature_names,
                                 class_names=data.target_names,
                                 discretize_continuous=True)

In [15]:
# Scelta di un'istanza da spiegare
instance_idx = 25  # Modifica questo indice per esplorare altre predizioni
instance = xtest[instance_idx]

# Generazione della spiegazione con LIME
exp = explainer.explain_instance(data_row=instance,
                                 predict_fn=mlp.predict_proba,
                                 num_features=10,  # Numero massimo di caratteristiche da visualizzare nella spiegazione
                                 top_labels=1)

In [16]:
!mkdir _aux

In [18]:
# Salviamo tutto per analisi successive

#datapoints preprocessati
np.save('_aux/xtest_prepr',xtest)

#modello addesterato
dill.dump(mlp,open('_aux/MLP_model.pkl','wb'))

#explainer
dill.dump(explainer,open('_aux/lime.pkl','wb'))

#singola spiegazione
dill.dump(exp,open('_aux/lime_single_'+str(instance_idx)+'.pkl','wb'))

In [19]:
! ls _aux

lime.pkl  lime_single_25.pkl  MLP_model.pkl  xtest_prepr.npy
